In [ ]:
# from huggingface_hub import login

# login("your_token_here")

In [ ]:
from ultralytics import YOLO
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

import os
import cv2
# Gunakan GPU 1
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

HOME = '/mnt/extended-home/frzamzami/Skripsi/Skripsi-Fay/Final/'

# Penggunaan Model
model = YOLO(f'{HOME}YOLO-PPE-Final-Comparison/yolo11m_balanced/weights/best.pt')

source = [f'{HOME}PPE-Dataset-Balanced/test/images/00629_jpg.rf.23777a7f91e081d9534a470d8da24c39.jpg'] #Gambar yang ingin di deteksi

results = model.predict(
    source,
    show_conf=True,
    show_labels=True,
    line_width=2
)

decoded_labels = [
    "Boots", "Ear-protection", "Glass", "Glove", "Helmet", "Mask", "Person", "Vest"
]

detected_objects = []

i = 19

for result in results:
    boxes = result.boxes.cls
    masks = result.masks
    keypoints = result.keypoints
    probs = result.probs
    obb = result.obb
    
    # Simpan gambar dengan bounding boxes menggunakan plot()
    annotated_image = result.plot()
    image_path = f"{HOME}LLM-Detection-Results/detection/result_{i}.jpg"
    cv2.imwrite(image_path, annotated_image)
    print(f"Gambar disimpan: {image_path}")
    
    # Simpan koordinat deteksi ke file txt
    result.save_txt(txt_file=f"{HOME}LLM-Detection-Results/detection/result_{i}.txt")
    
    name_objects = []
    local_objects = []
    seen_labels = set()

    for r in result:
        for c in r.boxes.cls:
            label = decoded_labels[int(c)]
            if label not in seen_labels:
                name_objects.append(label)
                seen_labels.add(label)
    
    # Filter: hanya APD, exclude 'Person'
    ppe_only = [obj for obj in name_objects if obj != 'Person']

    if len(ppe_only) > 1:
        local_objects = ", ".join(ppe_only[:-1]) + ", dan " + ppe_only[-1]
    elif len(ppe_only) == 1:
        local_objects = ppe_only[0]
    else:
        local_objects = "tidak ada APD yang terdeteksi"
    
    i = i + 1
    
    detected_objects.append(local_objects)


0: 800x800 1 Boots, 1 Person, 66.4ms
Speed: 15.1ms preprocess, 66.4ms inference, 147.6ms postprocess per image at shape (1, 3, 800, 800)
Gambar disimpan: /mnt/extended-home/frzamzami/Skripsi/Skripsi-Fay/Final/LLM-Detection-Results/detection/result_19.jpg


In [3]:
print(detected_objects)

['Boots']


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import gc

llm_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct",
    "unsloth/Qwen2.5-7B-Instruct",    
    "unsloth/Mistral-Nemo-Instruct-2407",
    "SulthanAbiyyu/llama3-cendol-sft",
    "Bahasalab/Bahasa-4b-chat",
    "kalisai/Nusantara-1.8B-Indo-Chat",
]

def load_model_and_tokenizer(model_name):
    """Load model and tokenizer with error handling"""
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, 
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True
        )
        
        # Set pad token if not exists
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            
        return model, tokenizer
    except Exception as e:
        print(f"Error loading {model_name}: {e}")
        return None, None

def generate_response(model, tokenizer, prompt, model_name):
    """Generate response using the model"""
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        
        if "Llama" in model_name:
            # Llama 3 format
            messages = [
                {"role": "system", "content": "Anda adalah asisten keselamatan kerja yang ahli dalam menganalisis penggunaan Alat Pelindung Diri (APD) di tempat kerja. Tugas Anda adalah mengevaluasi kelengkapan APD, menjelaskan fungsi masing-masing APD, dan memberikan peringatan keselamatan."},
                {"role": "user", "content": prompt}
            ]
            formatted_prompt = tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            
            model_inputs = tokenizer([formatted_prompt], return_tensors="pt")
            if torch.cuda.is_available():
                model_inputs = model_inputs.to(device)
            
            with torch.no_grad():
                output = model.generate(
                    model_inputs.input_ids,
                    attention_mask=model_inputs.attention_mask,
                    max_new_tokens=250,
                    eos_token_id=tokenizer.eos_token_id,
                    do_sample=True,
                    temperature=0.7,
                    top_k=50,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output)]
            generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
        elif "gemma" in model_name:
            # Gemma 3 format
            messages = [
                {"role": "user", "content": f"Anda adalah asisten keselamatan kerja yang ahli dalam menganalisis penggunaan Alat Pelindung Diri (APD) di tempat kerja. Tugas Anda adalah mengevaluasi kelengkapan APD, menjelaskan fungsi masing-masing APD, dan memberikan peringatan keselamatan.\n\n{prompt}"}
            ]
            
            formatted_prompt = tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            
            model_inputs = tokenizer([formatted_prompt], return_tensors="pt")
            if torch.cuda.is_available():
                model_inputs = model_inputs.to(device)
            
            with torch.no_grad():
                output = model.generate(
                    model_inputs.input_ids,
                    attention_mask=model_inputs.attention_mask,
                    max_new_tokens=250,
                    eos_token_id=tokenizer.eos_token_id,
                    do_sample=True,
                    temperature=0.7,
                    top_k=50,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output)]
            generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
        elif "Qwen" in model_name:
            # Qwen 3 format
            messages = [
                {"role": "system", "content": "Anda adalah asisten keselamatan kerja yang ahli dalam menganalisis penggunaan Alat Pelindung Diri (APD) di tempat kerja. Tugas Anda adalah mengevaluasi kelengkapan APD, menjelaskan fungsi masing-masing APD, dan memberikan peringatan keselamatan."},
                {"role": "user", "content": prompt}
            ]
            
            formatted_prompt = tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            
            model_inputs = tokenizer([formatted_prompt], return_tensors="pt")
            if torch.cuda.is_available():
                model_inputs = model_inputs.to(device)
            
            with torch.no_grad():
                output = model.generate(
                    model_inputs.input_ids,
                    attention_mask=model_inputs.attention_mask,
                    max_new_tokens=250,
                    eos_token_id=tokenizer.eos_token_id,
                    do_sample=True,
                    temperature=0.8,
                    top_k=50,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output)]
            generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
        elif "DeepSeek" in model_name:
            # DeepSeek R1 Distill format
            messages = [
                {"role": "system", "content": "Anda adalah asisten keselamatan kerja yang ahli dalam menganalisis penggunaan Alat Pelindung Diri (APD) di tempat kerja. Tugas Anda adalah mengevaluasi kelengkapan APD, menjelaskan fungsi masing-masing APD, dan memberikan peringatan keselamatan."},
                {"role": "user", "content": prompt}
            ]
            
            formatted_prompt = tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            
            model_inputs = tokenizer([formatted_prompt], return_tensors="pt")
            if torch.cuda.is_available():
                model_inputs = model_inputs.to(device)
            
            with torch.no_grad():
                output = model.generate(
                    model_inputs.input_ids,
                    attention_mask=model_inputs.attention_mask,
                    max_new_tokens=250,
                    eos_token_id=tokenizer.eos_token_id,
                    do_sample=True,
                    temperature=0.8,
                    top_k=50,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output)]
            generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
        else:
            # Universal format
            messages = [
                {"role": "system", "content": "Anda adalah asisten keselamatan kerja yang ahli dalam menganalisis penggunaan Alat Pelindung Diri (APD) di tempat kerja. Tugas Anda adalah mengevaluasi kelengkapan APD, menjelaskan fungsi masing-masing APD, dan memberikan peringatan keselamatan."},
                {"role": "user", "content": prompt}
            ]
            
            formatted_prompt = tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            
            model_inputs = tokenizer([formatted_prompt], return_tensors="pt")
            if torch.cuda.is_available():
                model_inputs = model_inputs.to(device)
            
            with torch.no_grad():
                output = model.generate(
                    model_inputs.input_ids,
                    attention_mask=model_inputs.attention_mask,
                    max_new_tokens=250,
                    eos_token_id=tokenizer.eos_token_id,
                    do_sample=True,
                    temperature=0.7,
                    top_k=50,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output)]
            generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
        return generated_text.strip()
        
    except Exception as e:
        print(f"Error generating response with {model_name}: {e}")
        return "Error generating response"

def clean_up_text(text):
    """Clean up generated text to ensure proper sentence completion"""
    if not text:
        return ""
    
    # Split into sentences
    sentences = text.split('. ')
    
    # Remove incomplete last sentence if it doesn't end with proper punctuation
    if sentences and not sentences[-1].endswith(('.', '!', '?')):
        sentences = sentences[:-1]
    
    if not sentences:
        return text
    
    # Join sentences back
    result = '. '.join(sentences)
    if not result.endswith(('.', '!', '?')):
        result += '.'
        
    return result

def clear_memory():
    """Clear GPU memory"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

def create_advanced_prompt(detected_items):
    """
    Buat prompt yang sophisticated berdasarkan analisis APD dengan prioritas
    """
    # APD Prioritas (WAJIB)
    priority_ppe = ['Vest', 'Boots', 'Helmet', 'Glove']
    
    # APD Kondisional (digunakan di situasi tertentu, bukan prioritas utama)
    conditional_ppe = ['Ear-protection', 'Glass', 'Mask']
    
    # Semua APD
    all_ppe = priority_ppe + conditional_ppe
    
    # Cek apakah ada Person
    has_person = 'Person' in detected_items
    
    # APD yang terdeteksi (exclude Person)
    detected_ppe = [item for item in detected_items if item != 'Person']
    
    # APD yang tidak terdeteksi (missing)
    missing_ppe = [item for item in all_ppe if item not in detected_ppe]
    
    # Klasifikasi APD yang terdeteksi
    detected_priority = [item for item in detected_ppe if item in priority_ppe]
    detected_conditional = [item for item in detected_ppe if item in conditional_ppe]
    
    # Klasifikasi APD yang hilang
    missing_priority = [item for item in priority_ppe if item not in detected_ppe]
    missing_conditional = [item for item in conditional_ppe if item not in detected_ppe]
    
    # Mapping nama APD ke penjelasan singkat
    ppe_descriptions = {
        'Vest': 'rompi keselamatan',
        'Boots': 'sepatu pelindung',
        'Helmet': 'helm pelindung kepala',
        'Glove': 'sarung tangan pelindung',
        'Ear-protection': 'pelindung telinga',
        'Glass': 'pelindung mata',
        'Mask': 'masker pelindung pernapasan'
    }
    
    # SKENARIO 1: Person terdeteksi dengan APD PRIORITAS LENGKAP
    if has_person and len(missing_priority) == 0:
        if len(missing_conditional) == 0:
            # SEMUA APD lengkap
            if len(detected_ppe) > 1:
                ppe_list = ", ".join(detected_ppe[:-1]) + ", dan " + detected_ppe[-1]
            else:
                ppe_list = detected_ppe[0]
            
            prompt = f"""Terdeteksi pekerja menggunakan APD lengkap: {ppe_list}. 

Berikan respons yang mencakup:
- Apresiasi bahwa pekerja sudah menggunakan semua APD dengan lengkap
- Jelaskan fungsi dari masing-masing APD yang digunakan untuk keselamatan kerja
- Jelaskan risiko jika tidak menggunakan APD tersebut

Tulis dengan bahasa yang natural dan sopan."""
        
        else:
            # APD prioritas lengkap, kondisional belum lengkap
            if len(detected_ppe) >= 3:
                # Apresiasi jika sudah pakai 3-4 APD atau lebih
                if len(detected_ppe) > 1:
                    ppe_detected_list = ", ".join(detected_ppe[:-1]) + ", dan " + detected_ppe[-1]
                else:
                    ppe_detected_list = detected_ppe[0]
                
                # APD yang tidak digunakan dengan deskripsi
                missing_with_desc = [f"{item} ({ppe_descriptions[item]})" for item in missing_conditional]
                if len(missing_with_desc) > 1:
                    missing_desc_list = ", ".join(missing_with_desc[:-1]) + ", dan " + missing_with_desc[-1]
                else:
                    missing_desc_list = missing_with_desc[0]
                
                prompt = f"""Kami mendeteksi bahwa pekerja sudah menggunakan APD yang cukup lengkap: {ppe_detected_list}. Namun, ada beberapa APD lainnya yang tidak digunakan, yaitu {missing_desc_list}.

Berikan respons yang mencakup:
- Apresiasi bahwa pekerja sudah menggunakan APD prioritas dengan baik
- Sebutkan APD yang tidak digunakan adalah {missing_desc_list}
- Jelaskan fungsi dan risiko tidak menggunakan masing-masing APD yang belum dipakai tersebut (satu per satu)
- Jelaskan juga fungsi dari APD yang sudah digunakan sebagai pelindung

Tulis dengan bahasa yang natural dan sopan."""
            else:
                # Kurang dari 3 APD
                if len(detected_ppe) > 1:
                    ppe_detected_list = ", ".join(detected_ppe[:-1]) + ", dan " + detected_ppe[-1]
                else:
                    ppe_detected_list = detected_ppe[0]
                
                missing_with_desc = [f"{item} ({ppe_descriptions[item]})" for item in missing_conditional]
                if len(missing_with_desc) > 1:
                    missing_desc_list = ", ".join(missing_with_desc[:-1]) + ", dan " + missing_with_desc[-1]
                else:
                    missing_desc_list = missing_with_desc[0]
                
                prompt = f"""Terdeteksi pekerja menggunakan APD: {ppe_detected_list}. APD yang tidak digunakan: {missing_desc_list}.

Berikan respons yang mencakup:
- Sebutkan APD yang tidak digunakan adalah {missing_desc_list}
- Jelaskan fungsi dan risiko tidak menggunakan masing-masing APD yang belum dipakai tersebut (satu per satu)
- Jelaskan fungsi dari APD yang sudah digunakan sebagai pelindung

Tulis dengan bahasa yang natural dan sopan."""
    
    # SKENARIO 2: Person terdeteksi TANPA APD PRIORITAS lengkap (PERINGATAN!)
    elif has_person and len(missing_priority) > 0:
        if len(detected_ppe) > 0:
            ppe_detected_list = ", ".join(detected_ppe[:-1]) + ", dan " + detected_ppe[-1] if len(detected_ppe) > 1 else detected_ppe[0]
        else:
            ppe_detected_list = "tidak ada"
        
        # Gabungkan missing prioritas dan kondisional
        all_missing = missing_priority + missing_conditional
        missing_with_desc = [f"{item} ({ppe_descriptions[item]})" for item in all_missing]
        if len(missing_with_desc) > 1:
            missing_desc_list = ", ".join(missing_with_desc[:-1]) + ", dan " + missing_with_desc[-1]
        else:
            missing_desc_list = missing_with_desc[0]
        
        prompt = f"""Peringatan Keselamatan:

Kami mendeteksi bahwa pekerja hanya menggunakan {ppe_detected_list}. Namun, ada beberapa APD lainnya yang tidak digunakan, yaitu {missing_desc_list}.

Berikan respons yang mencakup:
- Peringatan bahwa pekerja tidak menggunakan APD lengkap, sehingga berpotensi menghadapi risiko kecelakaan dan cedera
- Sebutkan APD yang tidak digunakan adalah {missing_desc_list}
- Jelaskan fungsi dan risiko tidak menggunakan masing-masing APD yang belum dipakai tersebut (satu per satu, terutama APD prioritas)
- Jelaskan fungsi dari APD yang sudah dipakai (jika ada) sebagai pelindung

Tulis dengan bahasa yang natural, tegas namun sopan."""
    
    # SKENARIO 3: TIDAK ada Person, hanya APD saja (education mode)
    elif not has_person and len(detected_ppe) > 0:
        if len(detected_ppe) > 1:
            ppe_list = ", ".join(detected_ppe[:-1]) + ", dan " + detected_ppe[-1]
        else:
            ppe_list = detected_ppe[0]
        
        prompt = f"""Terdeteksi APD yang tersedia: {ppe_list} (tanpa pekerja yang menggunakannya).

Berikan respons edukatif yang mencakup:
- Jelaskan fungsi dan pentingnya masing-masing APD ini untuk keselamatan kerja
- Sebutkan kapan dan di situasi apa APD ini harus digunakan
- Jelaskan risiko jika APD ini tidak digunakan saat bekerja

Tulis dengan bahasa yang natural dan sopan."""
    
    # SKENARIO 4: Tidak ada deteksi sama sekali
    else:
        prompt = """Tidak terdeteksi pekerja maupun APD dalam gambar.

Berikan penjelasan umum yang mencakup:
- Pentingnya selalu menggunakan APD lengkap di tempat kerja
- Sebutkan jenis-jenis APD prioritas (Vest, Boots, Helmet, Glove) dan kondisional (Ear-protection, Glass, Mask)
- Jelaskan fungsi masing-masing dan konsekuensi tidak menggunakannya

Tulis dengan bahasa yang natural dan sopan."""
    
    return prompt, has_person, detected_ppe, missing_ppe, detected_priority, missing_priority, detected_conditional, missing_conditional

print("Starting LLM pipeline with advanced safety analysis...")
print(f"Available models: {llm_models}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Proses setiap hasil deteksi
for idx, result in enumerate(results):
    print(f"\n{'='*80}")
    print(f"ANALISIS GAMBAR #{idx+1}")
    print(f"{'='*80}")
    
    # Ambil semua objek yang terdeteksi (termasuk Person)
    all_detected = []
    seen_labels = set()
    
    for r in result:
        for c in r.boxes.cls:
            label = decoded_labels[int(c)]
            if label not in seen_labels:
                all_detected.append(label)
                seen_labels.add(label)
    
    # Buat prompt yang advanced dengan prioritas APD
    full_prompt, has_person, detected_ppe, missing_ppe, detected_priority, missing_priority, detected_conditional, missing_conditional = create_advanced_prompt(all_detected)
    
    # Print analisis detail
    print(f"\n📋 DETEKSI:")
    print(f"   Pekerja (Person): {'✓ Ya' if has_person else '✗ Tidak'}")
    print(f"   APD Prioritas Terdeteksi: {', '.join(detected_priority) if detected_priority else 'Tidak ada'}")
    print(f"   APD Kondisional Terdeteksi: {', '.join(detected_conditional) if detected_conditional else 'Tidak ada'}")
    print(f"   APD Prioritas Tidak Dipakai: {', '.join(missing_priority) if missing_priority else 'Tidak ada (Lengkap!)'}")
    print(f"   APD Kondisional Tidak Dipakai: {', '.join(missing_conditional) if missing_conditional else 'Tidak ada (Lengkap!)'}")
    
    if has_person and len(missing_priority) > 0:
        print(f"\n   ⚠️⚠️ STATUS: PERINGATAN KERAS - APD PRIORITAS TIDAK LENGKAP!")
    elif has_person and len(missing_priority) == 0 and len(missing_conditional) == 0:
        print(f"\n   ✓✓ STATUS: SANGAT AMAN - APD LENGKAP (Prioritas + Kondisional)")
    elif has_person and len(missing_priority) == 0 and len(missing_conditional) > 0:
        print(f"\n   ✓ STATUS: AMAN - APD PRIORITAS LENGKAP (Sarankan APD Kondisional)")
    else:
        print(f"\n   ℹ️  STATUS: EDUKATIF - Tidak ada pekerja")
    
    print(f"\n📝 PROMPT:")
    print(f"{'-'*80}")
    print(full_prompt)
    print(f"{'-'*80}")
    
    # Generate response dari setiap LLM
    for model_name in llm_models:
        print(f"\n🤖 Processing with {model_name}...")
        
        try:
            # Load model and tokenizer
            model, tokenizer = load_model_and_tokenizer(model_name)
            
            if model is None or tokenizer is None:
                print(f"   ✗ Skipping {model_name} due to loading error")
                continue
            
            # Generate response
            generated_text = generate_response(model, tokenizer, full_prompt, model_name)
            
            # Clean up text
            cleaned_text = clean_up_text(generated_text)
            
            print(f"   ✓ Response: {cleaned_text}")
            
            # Clear memory after each model
            del model, tokenizer
            clear_memory()
            
        except Exception as e:
            print(f"   ✗ Error with {model_name}: {e}")
            clear_memory()
            continue

print("\n" + "="*80)
print("Pipeline completed!")
print("="*80)

Starting LLM pipeline with advanced safety analysis...
Available models: ['unsloth/Llama-3.2-11B-Vision-Instruct', 'unsloth/Qwen2.5-7B-Instruct', 'unsloth/Mistral-Nemo-Instruct-2407', 'SulthanAbiyyu/llama3-cendol-sft', 'Bahasalab/Bahasa-4b-chat', 'kalisai/Nusantara-1.8B-Indo-Chat']
CUDA available: True

ANALISIS GAMBAR #1

📋 DETEKSI:
   Pekerja (Person): ✓ Ya
   APD Prioritas Terdeteksi: Boots
   APD Kondisional Terdeteksi: Tidak ada
   APD Prioritas Tidak Dipakai: Vest, Helmet, Glove
   APD Kondisional Tidak Dipakai: Ear-protection, Glass, Mask

   ⚠️⚠️ STATUS: PERINGATAN KERAS - APD PRIORITAS TIDAK LENGKAP!

📝 PROMPT:
--------------------------------------------------------------------------------
Peringatan Keselamatan:

Kami mendeteksi bahwa pekerja hanya menggunakan Boots. Namun, ada beberapa APD lainnya yang tidak digunakan, yaitu Vest (rompi keselamatan), Helmet (helm pelindung kepala), Glove (sarung tangan pelindung), Ear-protection (pelindung telinga), Glass (pelindung mata), 

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

   ✓ Response: Peringatan Keselamatan:

Dengan hormat, saya ingin menarik perhatian Anda bahwa kami telah mendeteksi bahwa pekerja hanya menggunakan Boots (sepatu pelindung) dalam melakukan aktivitas di tempat kerja. Sayangnya, beberapa Alat Pelindung Diri (APD) lainnya tidak digunakan, yaitu Vest (rompi keselamatan), Helmet (helm pelindung kepala), Glove (sarung tangan pelindung), Ear-protection (pelindung telinga), Glass (pelindung mata), dan Mask (masker pelindung pernapasan).

Kami khawatir bahwa tidak menggunakan APD lengkap dapat meningkatkan risiko kecelakaan dan cedera di tempat kerja. Oleh karena itu, kami ingin menekankan pentingnya menggunakan APD yang sesuai dengan jenis pekerjaan dan kondisi kerja.

Berikut beberapa APD yang tidak digunakan dan risiko yang terkait:

1.

🤖 Processing with unsloth/Qwen2.5-7B-Instruct...


...

   ✓ Response: Pertama-tama, kami mengingatkan bahwa pekerja tidak menggunakan APD secara lengkap, yang meningkatkan risiko kecelakaan dan cedera. Terdapat beberapa APD yang tidak digunakan, yaitu Vest (rompi keselamatan), Helmet (helm pelindung kepala), Glove (sarung tangan pelindung), Ear-protection (pelindung telinga), Glass (pelindung mata), dan Mask (masker pelindung pernapasan).

Terlebih lagi, Vest (rompi keselamatan) bertindak sebagai pelindung tubuh dari gangguan dan kerusakan, mencegah cedera ringan dan serius. Tanpa menggunakan vest, pekerja berada di bawah risiko cedera yang lebih besar.

Helmet (helm pelindung kepala) adalah pelindung utama untuk kepala, melindungi pekerja dari hentakan, peluru, dan objek berat.

🤖 Processing with Bahasalab/Bahasa-4b-chat...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   ✓ Response: Peringatan Keselamatan:

 Kami menemukan bahwa beberapa pekerja tidak menggunakan Alat Pelindung Diri (APD) lengkap di tempat kerja.

🤖 Processing with kalisai/Nusantara-1.8B-Indo-Chat...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   ✓ Response: Pekerja hanya menggunakan boots, tetapi terdapat beberapa APD yang tidak digunakan, yaitu vest (rompi keselamatan), helm pelindung kepala, glove (sarung tangan pelindung), ear-protection (pelindung telinga), glass (pelindung mata), dan masker pelindung pernapasan. 

Fungsi dari APD yang sudah dipakai adalah pelindung telinga, karena dapat melindungi telinga dari cedera atau kecelakaan. APD yang belum dipakai adalah vest, helm pelindung kepala, glove, dan ear-protection. 

Fungsi dari APD yang belum dipakai adalah pelindung telinga, karena dapat melindungi telinga dari cedera atau kecelakaan. Vest, helm pelindung kepala, glove, dan masker pelindung pernapasan adalah APD yang harus digunakan sebagai pelindung. 

Kesimpulannya, pekerja harus menggunakan APD lengkap, termasuk vest, helm pelindung kepala, glove, dan ear-protection.

Pipeline completed!


with Appreciate

ppe_0137_jpg.rf.fd9654f350975fe81c8a887a7f111e97_aug1_342.jpg

Vest Saja

Twalv1052_jpg.rf.4067c606143eeec7d90201153570693d.jpg

Sebagian APD terpakai

TNPDu_78_jpg.rf.1de548b9ba8c39868559976e1f83a685.jpg

Tanpa APD 

00629_jpg.rf.23777a7f91e081d9534a470d8da24c39.jpg